In [ ]:
from pathlib import Path
%pip install numpy pandas
import numpy as np
import pandas as pd

LAST_NAME = "Quintela"
STUDENT_ID = "TUPM-26-3488"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

AssertionError: Missing file: c:\CODING\Lab7_NumPy_Pandas_Quintela\data\secom.data

In [ ]:
demo = np.array([2, 4, 6, 8,])

print("Sample Array:", demo)
print("Element-wise Square:", demo * 3)

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

print("shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())

In [ ]:
block = raw_measurements[:6, :4]
snapshot = block.copy()
block_copy = block.copy()

print("First Record:", block[0, :])
print("First Feature:", block[:, 0])
print("Last Two Records:\n", block[-2:, :])

reshaped = block_copy.reshape(3, 8)
print("Selected Block Shape:", block.shape)
print("Copied Block Shape:", block_copy.shape)
print("Reshaped Shape:", reshaped.shape)
print("Transposed Shape:", block.T.shape)
print("Flattened Size:", block.flatten().size)

block_copy[0, 0] = 999.0
print("Original Unchanged:",
      np.array_equal(block, snapshot, equal_nan=True))

In [ ]:
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

raw_usable = raw_measurements[:, usable_indices].copy()

feature_means = np.nanmean(raw_usable, axis=0)
feature_sds = np.nanstd(raw_usable, axis=0, ddof=1)

print("Total Missing Values:", missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Retained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

In [ ]:
missing_before = np.isnan(raw_usable).sum()
feature_medians = np.nanmedian(raw_usable, axis=0)

prepared_np = np.where(
    np.isnan(raw_usable), feature_medians, raw_usable
)

means = prepared_np.mean(axis=0)
sds = prepared_np.std(axis=0, ddof=1)
safe_sds = np.where(sds > 0, sds, 1.0)

standardized_np = (prepared_np - means) / safe_sds
flags = np.abs(standardized_np) > 3

print("Missing Before:", missing_before)
print("Missing After:", np.isnan(prepared_np).sum())
print("First Feature Median:", f"{feature_medians[0]:.4f}")
print("Standardized Mean:",
      f"{standardized_np[:, 0].mean():.4f}")
print("Standardized SD:",
      f"{standardized_np[:, 0].std(ddof=1):.4f}")
print("Values Beyond ±3:", np.count_nonzero(flags))

In [ ]:
n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

print("DataFrame Shape:", df.shape)
print("Measurement Features:", n_features)
print("Pass Records:", (labels == -1).sum())
print("Fail Records:", (labels == 1).sum())

In [ ]:
df.info(verbose=False)

print(df[feature_names[::3]].describe().round(4))
print("Missing Values:", df[feature_names].isna().sum().sum())
print("Outcome Counts:\n", df["Outcome"].value_counts())

print("Using iloc:\n", df.iloc[:3, :4])
print("Using loc;\n",
      df.loc[:2, [feature_names[0], "Outcome"]])

print("Series:", type(df[feature_names[0]]))
print("DataFrame:", type(df[[feature_names[0]]]))
print("Selected Feature:", feature_names[0])

In [ ]:
without_all_missing = df.dropna(axis=1, how="all")
retained_features = [feature_names[i] for i in usable_indices]

clean = without_all_missing[retained_features].copy()
clean = clean.fillna(clean.median())

clean["Result"] = df["Result"]
clean["Outcome"] = df["Outcome"]
clean["Is_Fail"] = clean["Result"] == 1

match = np.allclose(
    clean[retained_features].to_numpy(), prepared_np
)

print("All-Missing Columns Removed:", all_missing.sum())
print("Retained Features:", len(retained_features))
print("Cleaned Shape:", clean.shape)
print("Remaining Missing Values:", clean.isna().sum().sum())
print("Is_Fail True Count:", clean["Is_Fail"].sum())
print("NumPy-Pandas Match:", match)

In [ ]:
group_features = retained_features[::3]

grouped_stats = clean.groupby("Outcome")[group_features].agg(
    ["mean", "std"]
)

first_feature = retained_features[0]

numpy_sd = prepared_np[:, 0].std(ddof=1)
pandas_sd = clean[first_feature].std(ddof=1)

print("Grouped Statistics:\n", grouped_stats.round(4))

print("Selected Feature:", first_feature)
print("Pass Mean:",
      f"{grouped_stats.loc['Pass', (first_feature, 'mean')]:.4f}")
print("Fail Mean:",
      f"{grouped_stats.loc['Fail', (first_feature, 'mean')]:.4f}")
print("NumPy SD:", f"{numpy_sd:.4f}")
print("Pandas SD:", f"{pandas_sd:.4f}")
print("SD Match:", np.isclose(numpy_sd, pandas_sd))

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

feature_positions = rng.choice(
    len(retained_features), size=5, replace=False
)

assigned_features = [
    retained_features[i] for i in feature_positions
]

sample_size = min(200 + ID_SUM, len(clean))
row_positions = rng.choice(
    len(clean), size=sample_size, replace=False
)

sample = clean.iloc[row_positions][
    assigned_features + ["Result", "Outcome", "Is_Fail"]
].copy()

sample.insert(
    0, "Example_ID", clean.index.to_numpy()[row_positions]
)

print("Random Seed:", RANDOM_SEED)
print("Assigned Features:", assigned_features)
print("Sample Size:", len(sample))
print(sample.head())

In [ ]:
feature_summary = (
    sample[assigned_features]
    .agg(["count", "mean", "std", "min", "max"])
    .T.rename_axis("Feature")
    .reset_index()
    .sort_values(by="std", ascending=False)
)

outcome_means = (
    sample.groupby("Outcome")[assigned_features]
    .mean().reset_index()
)

counts = sample["Outcome"].value_counts()
pass_count = counts.get("Pass", 0)
fail_count = counts.get("Fail", 0)
failure_percentage = 100 * fail_count / len(sample)

highest_feature = feature_summary.iloc[0]["Feature"]
highest_sd = feature_summary.iloc[0]["std"]

prefix = f"{LAST_NAME}_{STUDENT_ID}"

sample_path = OUTPUT_DIR / f"{prefix}_secom_sample.csv"
summary_path = OUTPUT_DIR / f"{prefix}_feature_summary.csv"
outcome_path = OUTPUT_DIR / f"{prefix}_outcome_means.csv"
report_path = OUTPUT_DIR / f"{prefix}_analysis_report.txt"

sample.to_csv(sample_path, index=False)
feature_summary.to_csv(summary_path, index=False)
outcome_means.to_csv(outcome_path, index=False)

print("Feature Summary:\n", feature_summary.round(4))
print("Outcome Means:\n", outcome_means.round(4))
print("Highest-SD Feature:", highest_feature)
print("Highest Sample SD:", f"{highest_sd:.4f}")
print("Pass:", pass_count, "| Fail:", fail_count)
print("Failure Percentage:", f"{failure_percentage:.2f}%")
print("Three CSV files exported.")

In [ ]:
loaded_sample = pd.read_csv(sample_path)
loaded_summary = pd.read_csv(summary_path)
loaded_outcomes = pd.read_csv(outcome_path)

# Verify the complete exported sample
pd.testing.assert_frame_equal(
    loaded_sample, sample.reset_index(drop=True),
    check_dtype=False, check_exact=False,
    rtol=1e-10, atol=1e-10
)

# Verify measurements against their original record IDs
original_rows = clean.loc[loaded_sample["Example_ID"]]

np.testing.assert_allclose(
    loaded_sample[assigned_features].to_numpy(),
    original_rows[assigned_features].to_numpy(),
    rtol=1e-10, atol=1e-10
)

assert np.array_equal(
    loaded_sample["Result"], original_rows["Result"]
)
assert np.array_equal(
    loaded_sample["Outcome"], original_rows["Outcome"]
)

# Verify both statistical summaries
for actual, expected in [
    (loaded_summary, feature_summary.reset_index(drop=True)),
    (loaded_outcomes, outcome_means)
]:
    pd.testing.assert_frame_equal(
        actual, expected, check_dtype=False,
        check_exact=False, rtol=1e-10, atol=1e-10
    )

# Generate text report
report = (
    "SECOM NUMERICAL DATA ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Verification Valure: {verification_value}\n"
    f"Sample Size: {len(sample)}\n"
    f"Assigned Features: {', '.join(assigned_features)}\n"
    f"Pass Records: {pass_count}\n"
    f"Fail Records: {fail_count}\n"
    f"Sample Failure: {failure_percentage:.2f}%\n"
    f"Highest-SD Feature: {highest_feature}\n"
    f"Highest Sample SD: {highest_sd:.4f}\n"
    "Verification: PASSED\n\n"
    "Interpretation Notes:\n"
    "- Median imputation supports descriptive analysis.\n"
    "- Raw variability does not imply feature importance.\n"
    "- Group differences do not establish failure causes.\n"
    "- Sample failure percentage is not prediction accuracy.\n"
)

report_path.write_text(report, encoding="utf-8")

print("Exported Records:", len(loaded_sample))
print("Original Record IDs Verified:", True)
print("Numerical Values Verified:", True)
print("Pass/Fail Outcomes Verified:", True)

for path in [sample_path, summary_path, outcome_path, report_path]:
    print(path.name, "Created:", path.is_file())